# 02 - Data Cleaning

This notebook takes the findings from `01-exploratory-data-analysis.ipynb` and turns them into a clean dataset for feature engineering.

The goal of this notebook is **not** to train a model yet.

The goal is to:

- keep only the selected raw features
- remove duplicate rows
- clean basic date issues
- clean text fields
- clean categorical fields
- create simple text-size features
- save the cleaned dataset for feature engineering

## 02-01 Imports and notebook settings

Import pandas library needed for data cleaning.

In [1]:
from pathlib import Path
import pandas as pd

## 02-02 Load the raw dataset

Load the same raw Jira dataset used in the EDA notebook.

Adjust the path if your file is stored somewhere else.

In [3]:
ticket_df = pd.read_csv("../jira_ticket_dataset.csv")

print(f"Raw rows: {ticket_df.shape[0]:,}")
print(f"Raw columns: {ticket_df.shape[1]:,}")



C:\Users\Omar\AppData\Local\Temp\ipykernel_34632\1595426308.py:1: DtypeWarning: Columns (0: customfield_12310921, 1: issuetype.subtask) have mixed types. Specify dtype option on import or set low_memory=False.
  ticket_df = pd.read_csv("../jira_ticket_dataset.csv")


Raw rows: 1,149,323
Raw columns: 37


## 02-03 (Before-Selecting)

Before we eliminate any unnecessary columns's in predicting an outcome, some of them are useful in determining whether a task was finished successfully or not.

Columns such as, resolution.name, status.name, and statusCategory tell us which tasks have been recorderd as done, and these are the tasks we are going to be studying.

In [4]:
inspect_features = [
    "resolution.name",
    "status.name",
    "statusCategory.name"
]

inspection_df = {
    col: pd.DataFrame({
        'Count': ticket_df[col].value_counts(),
        'Percentage (%)': ticket_df[col].value_counts(normalize=True) * 100
    })
    for col in inspect_features
}

display(inspection_df)

display(ticket_df.sample(n=10, random_state=42))

created_dates = pd.to_datetime(ticket_df["created"], errors="coerce")
resolution_dates = pd.to_datetime(ticket_df["resolutiondate"], errors="coerce")

completed_issue_mask = (
    ticket_df["statusCategory.name"].eq("Done")
    & created_dates.notna()
    & resolution_dates.notna()
    & (resolution_dates >= created_dates)
)

rows_before = len(ticket_df)
ticket_df = ticket_df.loc[completed_issue_mask].copy()
rows_removed = rows_before - len(ticket_df)

print(f"Rows before completed issue filtering: {rows_before:,}")
print(f"Rows after completed issue filtering: {len(ticket_df):,}")
print(f"Rows removed: {rows_removed:,}")

ticket_df.shape

{'resolution.name':                        Count  Percentage (%)
 resolution.name                             
 Fixed                 720175       76.075761
 Duplicate              47404        5.007527
 Won't Fix              45620        4.819073
 Not A Problem          26478        2.797006
 Invalid                22529        2.379853
 Cannot Reproduce       17848        1.885375
 Done                   17584        1.857488
 Incomplete             11486        1.213325
 Implemented             8035        0.848778
 Later                   7301        0.771242
 Abandoned               5209        0.550253
 Won't Do                3661        0.386730
 Resolved                3640        0.384512
 Not A Bug               3242        0.342469
 Auto Closed             2694        0.284581
 Information Provided    2177        0.229968
 Workaround               593        0.062642
 Works for Me             327        0.034543
 Pending Closed           301        0.031796
 Feedback Recei

,id,key,summary,resolution.id,resolution.description,resolution.name,priority.id,priority.name,labels,assignee,...,project.key,project.name,projectCategory.id,projectCategory.description,projectCategory.name,resolutiondate,watches.watchCount,created,updated,description
362921,12523653.0,SMXCOMP-905,ensure DONE status is set when cxf bc consumer...,1.0,A fix for this issue is checked into the tree ...,Fixed,3.0,Major,[],83fabe05,...,SMXCOMP,ServiceMix Components,10480.0,ServiceMix Enterprise Service Bus,ServiceMix,2011-09-20 07:03:49,0.0,2011-09-20 06:26:44,2011-09-20 07:03:49,this can prevent potential thread leak
431071,12722862.0,UIMA-3908,Review UIMA-AS support for running with embedd...,3.0,The problem is a duplicate of an existing issue.,Duplicate,3.0,Major,[],baaaf95c,...,UIMA,UIMA,10410.0,Apache UIMA,UIMA,2014-06-27 19:21:37,1.0,2014-06-20 22:39:19,2014-06-27 19:21:37,UIMA-AS needs to support embedded AMQ broker. ...
580054,13095949.0,LIVY-311,Set classpath in MiniCluster only when it is n...,1.0,A fix for this issue is checked into the tree ...,Fixed,3.0,Major,[],46920e83,...,LIVY,Livy,NaN,NaN,NaN,2017-02-16 01:19:18,1.0,2017-02-08 06:23:57,2017-02-16 01:19:18,NaN
164680,12704840.0,SPARK-859,Remove the executor count from the header,1.0,A fix for this issue is checked into the tree ...,Fixed,3.0,Major,[],3b73e472,...,SPARK,Spark,NaN,NaN,NaN,2013-08-06 21:57:27,1.0,2013-08-05 15:12:06,2013-08-06 21:57:27,NaN
994340,13585790.0,HUDI-7988,ListingBasedRollbackStrategy support logcompact,1.0,A fix for this issue is checked into the tree ...,Fixed,3.0,Major,['pull-request-available'],99fd0329,...,HUDI,Apache Hudi,NaN,NaN,NaN,2024-07-17 09:42:27,1.0,2024-07-15 08:06:35,2024-07-17 09:42:27,[1. https://github.com/apache/hudi/issues/1158...
745408,13174292.0,ZEPPELIN-3662,Remove method register in Interpreter,1.0,A fix for this issue is checked into the tree ...,Fixed,4.0,Minor,[],46920e83,...,ZEPPELIN,Zeppelin,14260.0,Apache Zeppelin,Zeppelin,2018-07-26 02:39:18,3.0,2018-07-25 09:47:54,2020-12-24 04:20:25,"It is not used any more, can be removed."
703214,13215676.0,FLINK-11606,"Translate the ""Distributed Runtime Environment...",1.0,A fix for this issue is checked into the tree ...,Fixed,3.0,Major,['pull-request-available'],27afef9e,...,FLINK,Flink,13360.0,Apache Flink,Flink,2019-06-24 04:28:26,2.0,2019-02-14 07:54:54,2019-06-24 04:29:19,The page url is https://ci.apache.org/projects...
165951,12668629.0,PIVOT-922,"Some renderers were not calling their ""toStrin...",1.0,A fix for this issue is checked into the tree ...,Fixed,4.0,Minor,['renderer' 'wtk'],f147b299,...,PIVOT,Pivot,10382.0,Pivot related projects,Pivot,2013-10-01 16:34:12,2.0,2013-09-14 12:26:58,2013-10-01 16:34:54,"Most Renderer instances have the required ""ren..."
711160,13264998.0,FLINK-14555,Streaming File Sink s3 end-to-end test stalls,3.0,The problem is a duplicate of an existing issue.,Duplicate,2.0,Critical,['test-stability'],NaN,...,FLINK,Flink,13360.0,Apache Flink,Flink,2019-11-06 09:58:52,4.0,2019-10-29 10:28:04,2019-11-06 09:58:53,[https://api.travis-ci.org/v3/job/603882577/lo...
141541,12803139.0,MENFORCER-132,Add missing rules to standard rules page and r...,1.0,A fix for this issue is checked into the tree ...,Fixed,4.0,Minor,[],fb9a63fb,...,MENFORCER,Maven Enforcer Plugin,10510.0,Apache Maven related projects,Maven,2012-06-01 15:43:28,2.0,2012-06-01 13:53:55,2012-07-08 21:02:10,The {{RequireNoRepositories}} is quite an impo...


Rows before completed issue filtering: 1,149,323
Rows after completed issue filtering: 945,103
Rows removed: 204,220


(945103, 37)

## 02-03 Keep selected project columns

The EDA notebook identified the following columns as useful for the first modeling version.

`created` and `resolutiondate` are kept so the feature engineering notebook can create the target, though they are not model input variables.

In [5]:
raw_feature_columns = [
    "summary",
    "description",
    "priority.name",
    "issuetype.name",
    "resolutiondate",
    "created",
]

missing_columns = []

for col in raw_feature_columns:
    if col not in ticket_df.columns:
        missing_columns.append(col)

if missing_columns:
    print(f"Missing expected columns: {missing_columns}")

clean_df = ticket_df[raw_feature_columns].copy()

print(f"Rows after selecting columns: {clean_df.shape[0]:,}")
print(f"Columns after selecting columns: {clean_df.shape[1]:,}")

clean_df.head()

Rows after selecting columns: 945,103
Columns after selecting columns: 6


,summary,description,priority.name,issuetype.name,resolutiondate,created
0,Update config browser to work with the new syntax,The config browser used Velocity calling the t...,Minor,Improvement,2005-01-01 07:50:46,2005-01-01 07:47:50
1,XALAN_C 1.9 or current do not build on Fedora ...,Two types of errors:\n1- runConfigure and conf...,Blocker,Bug,2004-12-30 05:30:36,2004-12-25 22:50:30
2,"Problem with ADD new post, and DELETE post.","When trying to add new post, I was getting nex...",Critical,Bug,2005-01-02 15:21:00,2005-01-01 13:52:46
5,XmlConfigurator should support namespaces othe...,"Currently, the XmlConfigurator assumes that th...",Major,Improvement,2005-01-03 10:21:28,2004-12-27 01:34:17
6,GroovyServlet will crash if parameters are pas...,If parameters are being passed to a groovy ser...,Major,Bug,2005-01-03 11:29:27,2004-12-28 18:50:52


## 02-04 Remove duplicate rows

Remove exact duplicate rows only.

In [6]:
rows_before = len(clean_df)

clean_df = clean_df.drop_duplicates().copy()

rows_after = len(clean_df)
rows_removed = rows_before - rows_after

print(f"Rows before duplicate removal: {rows_before:,}")
print(f"Rows after duplicate removal: {rows_after:,}")
print(f"Duplicate rows removed: {rows_removed:,}")

Rows before duplicate removal: 945,103
Rows after duplicate removal: 937,217
Duplicate rows removed: 7,886


## 02-05 Clean date fields

Apply only the basic date cleaning needed before feature engineering.

Further duration cleaning, feature extraction, and target creation occur in the preprocessing notebook.

In [7]:
for col in ["created", "resolutiondate"]:
    clean_df[col] = pd.to_datetime(clean_df[col], errors="coerce")

rows_before = len(clean_df)
clean_df = clean_df.dropna(subset=["created", "resolutiondate"]).copy()
rows_after = len(clean_df)
missing_or_invalid_rows_removed = rows_before - rows_after

rows_before = len(clean_df)
clean_df = clean_df[clean_df["resolutiondate"] >= clean_df["created"]].copy()
rows_after = len(clean_df)
impossible_date_rows_removed = rows_before - rows_after

print(f"Rows removed for missing or invalid dates: {missing_or_invalid_rows_removed:,}")
print(f"Rows removed for impossible date order: {impossible_date_rows_removed:,}")
print(f"Rows after date cleaning: {clean_df.shape[0]:,}")

Rows removed for missing or invalid dates: 0
Rows removed for impossible date order: 0
Rows after date cleaning: 937,217


## 02-06 Clean text fields

Clean the selected text columns lightly and create simple text-size features.

Cleaning steps:

- fill missing text with an empty string
- convert values to strings
- normalize whitespace
- create simple character-count and word-count features

In [8]:
text_columns = ["summary", "description"]

for col in text_columns:
    clean_col = f"{col}"
    clean_df[clean_col] = (
        clean_df[col]
        .fillna("")
        .astype(str)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

clean_df["summary_char_count"] = clean_df["summary"].str.len()
clean_df["description_char_count"] = clean_df["description"].str.len()

clean_df["summary_word_count"] = clean_df["summary"].str.split().str.len()
clean_df["description_word_count"] = clean_df["description"].str.split().str.len()


clean_df[
    [
        "summary",
        "description",
        "summary_char_count",
        "description_char_count",
        "summary_word_count",
        "description_word_count",
    ]
].head()

,summary,description,summary_char_count,description_char_count,summary_word_count,description_word_count
0,Update config browser to work with the new syntax,The config browser used Velocity calling the t...,49,176,9,28
1,XALAN_C 1.9 or current do not build on Fedora ...,Two types of errors: 1- runConfigure and confi...,52,3445,11,206
2,"Problem with ADD new post, and DELETE post.","When trying to add new post, I was getting nex...",43,2560,8,124
5,XmlConfigurator should support namespaces othe...,"Currently, the XmlConfigurator assumes that th...",66,848,7,138
6,GroovyServlet will crash if parameters are pas...,If parameters are being passed to a groovy ser...,65,419,10,65


## 02-07 Clean categorical fields

Clean only the selected categorical columns.

Cleaning steps:

- fill missing values with `"Unknown"`
- normalize whitespace


In [9]:
categorical_columns = [
    "priority.name",
    "issuetype.name",
]

for col in categorical_columns:
    clean_col = col.replace(".", "_")
    clean_df[clean_col] = (
        clean_df[col]
        .fillna("Unknown")
        .astype(str)
        .str.strip()
        .replace("", "Unknown")
    )

clean_df[
    [
        "priority_name",
        "issuetype_name",
    ]
].head()

,priority_name,issuetype_name
0,Minor,Improvement
1,Blocker,Bug
2,Critical,Bug
5,Major,Improvement
6,Major,Bug


## 02-08 Check cleaned missing values

Inspect the cleaned fields before saving.

In [10]:
cleaned_model_columns = [
    "summary",
    "description",
    "priority_name",
    "issuetype_name",
    "summary_char_count",
    "summary_word_count",
    "description_char_count",
    "description_word_count",
    "created",
    "resolutiondate",
]

cleaned_missing_summary = pd.DataFrame({
    "missing_count": clean_df[cleaned_model_columns].isna().sum(),
    "missing_percent": clean_df[cleaned_model_columns].isna().mean().mul(100).round(2),
})

cleaned_missing_summary.sort_values("missing_percent", ascending=False)

,missing_count,missing_percent
summary,0,0.0
description,0,0.0
priority_name,0,0.0
issuetype_name,0,0.0
summary_char_count,0,0.0
summary_word_count,0,0.0
description_char_count,0,0.0
description_word_count,0,0.0
created,0,0.0
resolutiondate,0,0.0


## 02-09 Create final cleaned dataframe

Create a final dataframe containing only the cleaned columns needed for feature engineering.

In [11]:
model_df = clean_df[cleaned_model_columns].copy()
model_df_sample = model_df.sample(n=100, random_state=42)

print(f"Final cleaned rows: {model_df.shape[0]:,}")
print(f"Final cleaned columns: {model_df.shape[1]:,}")

Final cleaned rows: 937,217
Final cleaned columns: 10


## 02-10 Save cleaned dataset

Save the cleaned dataset for the feature engineering notebook.

In [12]:
output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

csv_path = output_dir / "jira_issues_cleaned.csv"
csv_sample_path = output_dir / "jira_issues_cleaned_sample.csv"

model_df.to_csv(csv_path)
model_df_sample.to_csv(csv_sample_path)
print(f"Saved CSV file to: {csv_path}")
print(f"Saved Sample CSV file to: {csv_sample_path}")

Saved CSV file to: ..\data\processed\jira_issues_cleaned.csv
Saved Sample CSV file to: ..\data\processed\jira_issues_cleaned_sample.csv


## 02-11 Cleaning summary

This notebook produced a clean dataset for feature engineering.

Cleaning actions completed:

- selected project-relevant features
- removed duplicate rows
- cleaned missing, invalid, and impossible date values
- cleaned text fields
- created simple text-size features
- cleaned categorical fields
- saved the cleaned dataset

Further duration cleaning, feature extraction, and target creation occur in the preprocessing notebook.